In [3]:
from torch import nn

class VanillaSkipgram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size, 
            embedding_dim= embedding_dim
        )

        self.linear = nn.Linear(
            in_features=embedding_dim, 
            out_features=vocab_size
        )
    
    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        output = self.linear(embeddings)
        return output

In [4]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load('nsmc')
corpus = pd.DataFrame(corpus.test)

tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /Users/janghyoin/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /Users/jan

In [5]:
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

vocab = build_vocab(corpus = tokens, n_vocab = 5000, special_tokens = ["<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [6]:
def get_word_pairs(tokens, window_size):
    pairs = []
    for sentence in tokens:
        sentence_length = len(sentence)
        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx - window_size)
            window_end = min(sentence_length, idx + window_size + 1)
            center_word = sentence[idx]
            context_words = sentence[window_start: idx] + sentence[idx + 1: window_end]
            for context_word in context_words:
                pairs.append([center_word, context_word])
    return pairs

word_pairs = get_word_pairs(tokens, window_size=2)
print(word_pairs[:5])

[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


In [7]:
def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id["<unk>"]
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs

index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]


In [8]:
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs = torch.tensor(index_pairs)
center_indexes = index_pairs[:, 0]
context_indexes = index_pairs[:, 1]

dataset = TensorDataset(center_indexes, context_indexes)
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True)

In [9]:
from torch import optim

device = 'mps' if torch.backends.mps.is_available() else "cpu"
word2vec = VanillaSkipgram(vocab_size=len(token_to_id), embedding_dim=128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr = 0.1)

In [13]:
for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss

    cost = cost / len(dataloader)
    print(f"Epoch: {epoch+1:4d}, Cost: {cost:.3f}")

Epoch:    1, Cost: 5.982
Epoch:    2, Cost: 5.933
Epoch:    3, Cost: 5.903
Epoch:    4, Cost: 5.881
Epoch:    5, Cost: 5.863
Epoch:    6, Cost: 5.848
Epoch:    7, Cost: 5.836
Epoch:    8, Cost: 5.824
Epoch:    9, Cost: 5.814
Epoch:   10, Cost: 5.804


In [14]:
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()

for word, embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding

index = 30
token = vocab[30]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)


연기
[-1.9815925  -0.27938595 -0.66432756  0.15459386 -0.8807213   0.14786002
  0.64498925  0.1671892  -0.2414137   0.04376874 -0.7302386   0.18936345
 -0.57505965  1.0381693  -0.5814681   0.33503428  1.2105416   0.744385
 -0.8410904   0.61983615 -0.21655789  0.9868787  -0.83648425 -0.39607635
 -0.7772915   1.1710385   0.08705818 -1.0495017  -0.9618623  -0.86243856
 -0.9551921   0.43525833  0.4038037   0.0091143   0.6162521   0.16072837
 -0.8632577  -1.0972307   1.1155233   1.7636108  -1.3620842   1.2533647
  0.6578477  -0.7646224   2.4733486  -1.46763     0.6717474   0.6240405
 -0.59723425 -0.01888536  1.5286694   0.78765893  2.1047812  -0.53783995
  0.32375962  0.06312134  1.4547968  -0.3007019   0.17569126 -0.40104353
 -1.6516781   0.51341116 -0.20451538 -0.9581142   0.11553036 -0.15327851
 -0.03467102 -0.18076742  0.55899113 -1.0317096   1.0995387  -0.35254836
 -0.43045923  1.1635298   0.17001708 -1.7297022   0.8083983  -2.111588
  0.899763    0.9920264  -0.51245517 -0.9879903   0.98

In [24]:
import numpy as np
from numpy.linalg import norm # euclid norm

def cosine_similarity(a, b):
    cosine = np.dot(b, a) / (norm(b, axis = 1) * norm(a)) # 내적 / 두 벡터 크기의 곱
    return cosine

def top_n_index(cosine_matrix, n):
    closest_indexes = cosine_matrix.argsort()[::-1] # reverse
    top_n = closest_indexes[1: n+1]
    return top_n

cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n =5)

print(f"{token}와 가장 유사한 5 개 단어")
for index in top_n:
    print(f"{id_to_token[index]} - 유사도: {cosine_matrix[index]:.4f}")

연기와 가장 유사한 5 개 단어
연기력 - 유사도: 0.3214
여자 - 유사도: 0.3126
답답하다 - 유사도: 0.2931
황당한 - 유사도: 0.2860
고딩 - 유사도: 0.2720


In [9]:
from gensim.models import Word2Vec

word2vec = Word2Vec(
    sentences=tokens,  
    vector_size=128,
    window=5,
    min_count=1,
    sg=1,
    epochs=3,
    max_final_vocab=10000
)

In [14]:
word2vec.save("models/word2vec.model")
word2vec = Word2Vec.load("models/word2vec.model")

In [15]:
word = '연기'
print(word2vec.wv[word])
print(word2vec.wv.most_similar(word, topn = 5))
print(word2vec.wv.similarity(w1 = word, w2 = "연기력"))

[-0.32991582 -0.3077436   0.1018536   0.3394808  -0.10078396  0.07406776
 -0.08742826 -0.0783589  -0.5418248   0.5030264  -0.08281668 -0.31624553
  0.02647373 -0.00716113  0.23071702  0.06036479 -0.21342942  0.37393048
 -0.18596834  0.16234246  0.684702    0.26185456 -0.24195373 -0.19425535
 -0.18497355  0.08184844 -0.4752375   0.0449965   0.09078958 -0.19087994
 -0.53809595  0.0183689   0.38310072 -0.07961736 -0.03978975 -0.02633742
  0.01659325 -0.07892772 -0.0017791  -0.13347657  0.02801502 -0.03882441
 -0.38101333 -0.27940565 -0.27319863 -0.00205845 -0.39466995 -0.21730027
  0.12114257  0.2191869   0.42713597  0.35757875  0.23723166  0.14919747
 -0.4523237  -0.0657582   0.08176541  0.27661917 -0.25127667  0.15919068
 -0.00791723 -0.22857046  0.2043587   0.2297673  -0.21646923  0.1594185
 -0.09707396  0.33163607  0.2369715  -0.29150516 -0.48337558 -0.27094063
 -0.3725709   0.20694615  0.08468869 -0.28375745 -0.25209275 -0.22031595
 -0.02853148  0.17922851 -0.25525835 -0.09100151  0.

In [17]:
from Korpora import Korpora

corpus = Korpora.load("kornli")
corpus_text = corpus.get_all_texts() + corpus.get_all_pairs()
tokens = [sentence.split() for sentence in corpus_text]

print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : KakaoBrain
    Repository : https://github.com/kakaobrain/KorNLUDatasets
    References :
        - Ham, J., Choe, Y. J., Park, K., Choi, I., & Soh, H. (2020). KorNLI and KorSTS: New Benchmark
           Datasets for Korean Natural Language Understanding. arXiv preprint arXiv:2004.03289.
           (https://arxiv.org/abs/2004.03289)

    This is the dataset repository for our paper
    "KorNLI and KorSTS: New Benchmark Datasets for Korean Natural Language Understanding."
    (https://arxiv.org/abs/2004.03289)
    We introduce KorNLI and KorSTS, which are NLI and STS datasets in Korean.

    # License
    Creative Commons Attribution-ShareAlike license (CC BY-SA 4.0)
    Details in https://creativecommons.org/licenses

In [18]:
from gensim.models import FastText

fastText = FastText(
    sentences=tokens, 
    vector_size=128, 
    window = 5, 
    min_count = 5, 
    sg = 1, 
    epochs = 3, 
    min_n = 2, 
    max_n = 6
)

In [19]:
fastText.save("models/fastText.model")
fastText = FastText. load("models/fastText.model")

In [20]:
oov_token = "사랑해요"
oov_vector = fastText.wv[oov_token]

print(oov_token in fastText.wv.index_to_key)
print(fastText.wv.most_similar(oov_vector, topn = 5))

False
[('사랑해', 0.9065362811088562), ('사랑한', 0.8637875914573669), ('사랑', 0.861009955406189), ('사랑해서', 0.850717306137085), ('사랑해.', 0.8334926962852478)]


In [21]:
import torch
from torch import nn

input_size = 128
output_size = 256
num_layers = 3
bidirectional = True

model = nn.RNN(
    input_size=input_size, 
    hidden_size=output_size, 
    num_layers=num_layers, 
    nonlinearity='tanh', 
    batch_first=True, 
    bidirectional=bidirectional,
)

batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.rand(num_layers * (int(bidirectional) + 1), batch_size, output_size)

outputs, hidden = model(inputs, h_0)
print(outputs.shape)
print(hidden.shape)

torch.Size([4, 6, 512])
torch.Size([6, 4, 256])


In [22]:
import torch
from torch import nn

input_size = 128
ouput_size = 256
num_layers = 3
bidirectional = True
proj_size = 64

model = nn.LSTM(
    input_size = input_size,
    hidden_size = ouput_size,
    num_layers=num_layers,
    batch_first=True,
    bidirectional = bidirectional,
    proj_size = proj_size,
)

batch_size = 4
sequence_len = 6
inputs = torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.rand(
    num_layers * (int(bidirectional) + 1),
    batch_size,
    proj_size if proj_size > 0 else ouput_size,
)

c_0 = torch.rand(num_layers * (int(bidirectional) + 1), batch_size, output_size)

outputs, (h_n, c_n) = model(inputs, (h_0, c_0))

print(outputs.shape)
print(h_n.shape)
print(c_n.shape)

torch.Size([4, 6, 128])
torch.Size([6, 4, 64])
torch.Size([6, 4, 256])


In [40]:
from torch import nn
class SentenceClassifier(nn.Module):
    def __init__(
            self, 
            n_vocab, # 단어 사전 크기
            hidden_dim,
            embedding_dim,
            n_layers, 
            dropout = 0.5, 
            bidirectional = True, 
            model_type = 'lstm' # 모델 종류에 따라 RNN, LSTM 사용 여부 결정
    ): 
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=n_vocab,
            embedding_dim = embedding_dim,
            padding_idx=0
        )
        if model_type == "rnn":
            self.model = nn.RNN(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                bidirectional = bidirectional,
                dropout=dropout,
                batch_first=True,
            )
        elif model_type == "lstm":
            self.model = nn. LSTM(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                bidirectional=bidirectional,
                dropout=dropout,
                batch_first=True,
            )
        
        if bidirectional: 
        # 양방향으로 분류기 계층을 구성하면 전달되는 입력 채널 수가 달라지므로 분류기 계층을 조절
            self.classifier = nn.Linear(hidden_dim * 2, 1)
        else:
            self.classifier = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, inputs):
        embeddings = self.embedding(inputs) 
        # 입력 받은 정수 인코딩을 임베딩 계층에 통과시켜 임베딩 값을 얻음
        
        output,_ = self.model(embeddings)
        last_output = output[:, -1, :] # 출력값의 마지막 시점만 활용
        last_output = self.dropout(last_output)
        logits = self.classifier(last_output)
        return logits

In [27]:
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load("nsmc")
corpus_df = pd.DataFrame(corpus.test)

train = corpus_df.sample(frac = 0.9, random_state=42)
test = corpus_df.drop(train.index)

print(train.head(5).to_markdown())
print("Training Data Size:", len(train))
print("Testing Data Size:", len(test))


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /Users/janghyoin/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /Users/jan

In [31]:
from konlpy.tag import Okt
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update (tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

tokenizer = Okt()
train_tokens = [tokenizer.morphs(review) for review in train.text]
test_tokens = [tokenizer.morphs(review) for review in test.text]

vocab = build_vocab(corpus=train_tokens, n_vocab=5000, special_tokens= ["<pad>", "<unk>"])
# 문장 길이를 맞추기 위해 <pad> 토큰을 special_tokens에 추가
# 2 개의 특수 토큰과 단어 사전 최대 길이 5000개 -> 단어 사전 50002개

token_to_id = {token: idx for idx, token in enumerate(vocab) }
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<pad>', '<unk>', '.', '이', '영화', '의', '..', '가', '에', '...']
5002


In [32]:
import numpy as np

def pad_sequences(sequences, max_length, pad_value):
    result = list()
    for sequence in sequences:
        sequence = sequence[: max_length]
        pad_length = max_length - len(sequence)
        padded_sequence = sequence + [pad_value] * pad_length
        result.append(padded_sequence)
    return np.asarray(result)

unk_id = token_to_id["<unk>"]
train_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in train_tokens
]
test_ids = [
[token_to_id.get(token, unk_id) for token in review] for review in test_tokens
]

max_length = 32
pad_id = token_to_id["<pad>"]
train_ids = pad_sequences(train_ids, max_length, pad_id)
test_ids = pad_sequences(test_ids, max_length, pad_id)

print(train_ids[0])
print(test_ids[0])

[ 223 1716   10 4036 2095  193  755    4    2 2330 1031  220   26   13
 4839    1    1    1    2    0    0    0    0    0    0    0    0    0
    0    0    0    0]
[3307    5 1997  456    8    1 1013 3906    5    1    1   13  223   51
    3    1 4684    6    0    0    0    0    0    0    0    0    0    0
    0    0    0    0]


In [33]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_ids = torch.tensor(train_ids)
test_ids = torch.tensor(test_ids)

train_labels = torch. tensor(train.label.values, dtype=torch.float32)
test_labels = torch. tensor(test.label.values, dtype=torch.float32)

train_dataset = TensorDataset(train_ids, train_labels)
test_dataset = TensorDataset(test_ids, test_labels)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [41]:
from torch import optim

n_vocab = len(token_to_id)

hidden_dim = 64
embedding_dim = 128
n_layers= 2

device = "mps" if torch.backends.mps.is_available() else "cpu"

classifier = SentenceClassifier(
    n_vocab=n_vocab, 
    hidden_dim=hidden_dim, 
    embedding_dim = embedding_dim, 
    n_layers=n_layers
).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr=0.001)

In [42]:
# 학습 함수
def train(model, datasets, criterion, optimizer, device, interval):
    model.train()
    losses = list()

    for step, (input_ids, labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % interval == 0:
            print(f"Train Loss {step} : {np.mean(losses)}")


# 평가 함수
def test(model, datasets, criterion, device):
    model.eval()
    losses = list()
    corrects = list()

    for step, (input_ids, labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())

        yhat = torch.sigmoid(logits) > 0.5
        corrects.extend(torch.eq(yhat, labels).cpu().tolist())

    print(f"Val Loss : {np.mean(losses)}, Val Accuracy : {np.mean(corrects)}")


# 전체 학습 루프
epochs = 5
interval = 500

for epoch in range(epochs):
    train(classifier, train_loader, criterion, optimizer, device, interval)
    test(classifier, test_loader, criterion, device)


Train Loss 0 : 0.7030100226402283
Train Loss 500 : 0.6938743916100371
Train Loss 1000 : 0.6937089832750829
Train Loss 1500 : 0.6837930516709017
Train Loss 2000 : 0.6733403067062165
Train Loss 2500 : 0.6652066619431481
Val Loss : 0.618286716100126, Val Accuracy : 0.6262
Train Loss 0 : 0.5801323056221008
Train Loss 500 : 0.6116388830358159
Train Loss 1000 : 0.6109639644384622
Train Loss 1500 : 0.6115920151931933
Train Loss 2000 : 0.6046077690828686
Train Loss 2500 : 0.593769078681775
Val Loss : 0.5236749263426748, Val Accuracy : 0.7448
Train Loss 0 : 0.5280954837799072
Train Loss 500 : 0.4816389282544454
Train Loss 1000 : 0.4698582575930939
Train Loss 1500 : 0.46124584303745664
Train Loss 2000 : 0.45324592863482754
Train Loss 2500 : 0.44683753818523786
Val Loss : 0.4190034214585734, Val Accuracy : 0.8018
Train Loss 0 : 0.25675803422927856
Train Loss 500 : 0.3786410425237553
Train Loss 1000 : 0.3755786275932124
Train Loss 1500 : 0.3769880813669952
Train Loss 2000 : 0.37657121371948854
Tra

In [43]:
token_to_embedding = dict()
embedding_matrix = classifier.embedding.weight.detach().cpu().numpy()

for word, emb in zip(vocab, embedding_matrix):
    token_to_embedding[word] = emb

token = vocab [1000]
print(token, token_to_embedding[token])

보고싶다 [ 0.13531028 -0.31394956 -0.02311889 -1.1241776   1.1804674  -0.7845719
  1.3130592  -0.28796998 -1.7821255  -0.16553274  1.1856333  -0.646884
  1.3097075  -1.0348458  -0.37890163 -2.2617595  -1.4062423   1.33481
  1.6794492   0.22142035 -0.32608196 -0.25889197 -0.17325686 -0.6283211
  0.7197847   1.689144   -0.85693383 -0.43386778  0.42066893 -1.9965403
 -1.0185982  -0.01098159  0.7771058  -0.45929796 -1.5312567  -0.13361801
 -2.1635413  -0.26737309 -0.6540926   0.39472282 -0.4647599   0.01972922
 -1.1372222   1.5023367   2.796877   -1.2578715  -0.83512855  0.6189778
  0.1594828   0.4764562  -0.33699322  0.8030396  -0.47220162  0.30940038
 -1.1686894   0.00855597  0.68248576 -0.7079853  -1.6589357  -0.6170184
  1.6325111  -0.29134938 -2.4702597  -2.2351334   0.39243278 -0.33110848
 -0.0296989  -0.33925787  0.11178508  0.9033625   0.9200379  -1.0668489
 -1.551481    2.229732    2.2985795   1.1400802  -0.15997456 -0.7116491
 -0.5233224  -0.20572045  0.13059522  1.0335954   1.031513

In [48]:
from gensim.models import Word2Vec
word2vec = Word2Vec.load('models/word2vec.model')
init_embeddings = np. zeros ((n_vocab, embedding_dim))

for index, token in id_to_token. items():
    if token not in ["<pad>", "<unk>"]:
        init_embeddings[index] = word2vec.wv[token]

embedding_layer = nn. Embedding.from_pretrained(
torch. tensor (init_embeddings, dtype=torch.float32)
)

In [49]:
from torch import nn
class SentenceClassifier(nn.Module):
    def __init__(
            self, 
            n_vocab, # 단어 사전 크기
            hidden_dim,
            embedding_dim,
            n_layers, 
            dropout = 0.5, 
            bidirectional = True, 
            model_type = 'lstm', # 모델 종류에 따라 RNN, LSTM 사용 여부 결정
            pretrained_embedding=None 
    ): 
        super().__init__()
        if pretrained_embedding is not None:
            self.embedding = nn.Embedding.from_pretrained(
                torch.tensor(pretrained_embedding, dtype=torch.float32),
                freeze=False  # 필요 시 True로 설정하여 학습 중 임베딩 고정 가능
            )
        else:
            self.embedding = nn.Embedding(
                num_embeddings=n_vocab,
                embedding_dim=embedding_dim,
                padding_idx=0
            )

        if model_type == "rnn":
            self.model = nn.RNN(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                bidirectional = bidirectional,
                dropout=dropout,
                batch_first=True,
            )
        elif model_type == "lstm":
            self.model = nn. LSTM(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                bidirectional=bidirectional,
                dropout=dropout,
                batch_first=True,
            )
        
        if bidirectional: 
        # 양방향으로 분류기 계층을 구성하면 전달되는 입력 채널 수가 달라지므로 분류기 계층을 조절
            self.classifier = nn.Linear(hidden_dim * 2, 1)
        else:
            self.classifier = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, inputs):
        embeddings = self.embedding(inputs) 
        # 입력 받은 정수 인코딩을 임베딩 계층에 통과시켜 임베딩 값을 얻음
        output,_ = self.model(embeddings)
        last_output = output[:, -1, :] # 출력값의 마지막 시점만 활용
        last_output = self.dropout(last_output)
        logits = self.classifier(last_output)
        return logits

In [ ]:
classifier = SentenceClassifier(
    n_vocab=n_vocab, 
    hidden_dim=hidden_dim, 
    embedding_dim = embedding_dim, 
    n_layers=n_layers
).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr=0.001)

epochs = 5
interval = 500

for epoch in range(epochs):
    train(classifier, train_loader, criterion, optimizer, device, interval)
    test(classifier, test_loader, criterion, device)

Train Loss 0 : 0.6932474970817566
Train Loss 500 : 0.6938734870708869
Train Loss 1000 : 0.686015983353128
Train Loss 1500 : 0.6727352509253983
Train Loss 2000 : 0.6598642934029963
